# **[실습] 주택청약 FAQ 시스템 구현**

### **문제 설명**
이전 코드를 기반으로 주택청약 FAQ 시스템을 다음 요구사항에 맞춰 개선합니다. 

1. 응답 품질 향상 (1개 이상)
   - 생성된 답변의 품질을 평가 (답변이 불충분한 경우 예외 처리)
   - 관련성 높은 FAQ 문서 검색 (임베딩 모델, 청크 크기, 벡터 검색 방법 등) 
   / 메타데이터 필터링, 관련성 평가, 포매팅 고려하여 적용

2. 사용자 경험 개선 (1개 이상)
   - 대화 이력 관리 기능 추가 (요약, 트리밍 기능 등 고려)
   - 최근 대화 기반 컨텍스트 구성 
   - 사용자 프로필 기반 맞춤 응답

### **제약 조건**
- Gradio ChatInterface 사용
- RAG 구조 유지

** 데이터 파일**
- `data/housing_faq.txt` 파일 로드 (국토교통부 주택청약 FAQ 50개 Q&A)

- 'data/2024 주택청약 FAQ.pdf' 

# 환경 설정 및 준비

`(1) Env 환경변수`

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [2]:
import os
from glob import glob

from pprint import pprint
import json

`(3) LLM 설정`

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model='gpt-4.1-mini',      # 사용할 모델
    temperature=0.1,            # 낮은 값: 일관된 답변 (0.0~2.0)
    top_p=0.9,                  # 토큰 샘플링 확률 임계값 (0.0~1.0)
)

e:\sw\dev\ai\modu_llm7\faq_bot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 국토교통부 - 주택청약 FAQ 문서 업로드
 - https://www.molit.go.kr/USR/policyData/m_34681/dtl.jsp?search=&srch_dept_nm=&srch_dept_id=&srch_usr_nm=&srch_usr_titl=Y&srch_usr_ctnt=&search_regdate_s=&search_regdate_e=&psize=10&s_category=p_sec_2&p_category=&lcmspage=1&id=4765 의 pdf 파일 다운로드 


In [9]:
from langchain_community.document_loaders import PyMuPDFLoader

# 1. 문서 로드
pdf_path = './data/2024 주택청약 FAQ.pdf'
print("1단계: PDF 문서 로딩 중...")

loader = PyMuPDFLoader(pdf_path)
raw_documents = loader.load()

# 문서 개수 출력
print(f"PDF 문서 개수: {len(raw_documents)}")
print("-" * 50)

# 각 문서(페이지)의 텍스트 길이 출력
# for i, doc in enumerate(raw_documents):
#    print(f"페이지 {i+1}: {len(doc.page_content)} 글자")
    
# 전체 텍스트 길이
total_length = sum(len(doc.page_content) for doc in raw_documents)
print("-" * 50)
print(f"전체 텍스트 길이: {total_length} 글자")
print("-------" + raw_documents[34].page_content)

C:\Users\JSPark\AppData\Local\Temp\ipykernel_36260\2761222554.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


1단계: PDF 문서 로딩 중...
PDF 문서 개수: 273
--------------------------------------------------
--------------------------------------------------
전체 텍스트 길이: 263621 글자
-------Ⅰ. 청약자격(공통)
3
청약신청지역
1
가. 주요내용
●청약신청 지역을 판단하는 기준인 ‘주택건설지역’이란 주택이 건설되는 특별시ㆍ광역시ㆍ특별
자치시ㆍ특별자치도 또는 시ㆍ군의 행정구역을 뜻합니다.
●주택은 원칙적으로 입주자모집공고일 현재 해당 주택건설지역에 거주하고 있는 자를 공급대상
으로 정하고 있으나, 
● 제4조제1항제3호, 제4조제3항에 따른 청약가능지역에 거주하고 있는 경우 해당 주택건설지역에
거주하고 있지 않아도 청약신청이 가능합니다.
  * 청약가능지역 : 법령상 용어는 아니나, 특정 지역에서 주택을 공급하는 경우 청약이 가능한 지역을 
의미함
(예) 산업단지 : 전국에서 청약 가능, 서울 : 수도권 거주자 청약 가능
제2조(정의) 이 규칙에서 사용하는 용어의 뜻은 다음과 같다.
  2. “주택건설지역”이란 주택을 건설하는 특별시ㆍ광역시ㆍ특별자치시ㆍ특별자치도(관할 구역 
안에 지방자치단체인 시ㆍ군이 없는 특별자치도를 말한다) 또는 시ㆍ군의 행정구역을 말한다. 
이 경우 주택건설용지를 공급하기 위한 사업지구 등이 둘 이상의 특별시ㆍ광역시ㆍ특별자치시 
또는 시ㆍ군의 행정구역에 걸치는 경우에는 해당 행정구역 모두를 같은 주택건설지역으로 
본다
제4조(주택의 공급대상) ① 주택의 공급대상은 다음 각 호의 기준에 따른다.
  1. 국민주택과 제3조제2항제1호에 따른 주택은 입주자모집공고일 현재 해당 주택건설지역에 
거주하는 성년자인 무주택세대구성원에게 1세대 1주택[공급을 신청(「공공주택 특별법 시행
령」 제2조제1항제1호부터 제3호까지, 제3호의2, 제4호, 제6호 및 제7호에 따른 주택 외
의 주택으로서 당첨일이 같은 주택에 대해 부부가 각각 공급을 신청하는 경

In [10]:
import re
from langchain_core.documents import Document

start_page_idx = 0

for i, doc in enumerate(raw_documents):
    if "Ⅰ. 청약자격(공통)" in doc.page_content:
        print("문서 시작 부분", i)
       # print(doc.page_content)
        start_page_idx = i
        break

# 문서 구조상 33~34 페이지부터 본문 (인덱스 부분 제거)
if start_page_idx == 0:
    start_page_idx = 33 

print(start_page_idx)
print(raw_documents[50].page_content)


문서 시작 부분 34
34
Ⅰ. 청약자격(공통)
19
나. 주요 질의 및 답변
21
기존 청약통장을 주택청약종합저축으로 전환할 수 있나요?
기존 청약통장(청약저축, 청약부금, 청약예금)에서 주택청약종합저축으로의 전환은 불가능
하며, 기존 청약통장을 해지한 후 신규 가입으로만 가능합니다.
구  분
주 요 내 용
비과세
‣ (요건) 가입기간 2년 이상
‣ (한도) 총 이자소득의 500만원 및 납입원금 연 600만원까지 비과세 적용
  * 비과세 대상 및 요건은 조세특례제한법에 따르며, 연소득·무주택세대주 요건은 
우대이율 요건과 일부 다름
소득공제
‣ 기존 「주택청약종합저축」과 동일
  * 조세특례제한법 상의 요건을 충족한 경우 연소득 7천만원 이하이며 무주택
세대주인 경우 연간 300만원 한도로 40%까지
전환신규
‣ 기존 「주택청약종합저축」에서 「청년주택드림청약통장」으로 전환가능
  * 기존 가입기간 및 납입인정회차(선납 및 연체일수 등은 제외)를 연속하여 인정
‣ (방식) 기존통장 해지 후 전환원금을 신규통장으로 이전
  * 전환원금은 청약회차 및 우대이율에서 제외됨
증빙서류 
‣ (연령) 주민등록등본, (병역기간) 병적증명서
‣ (무주택) 각서(서식)
‣ (연소득) 소득확인증명서(상품 가입 및 과세특례 신청용) 등 
‣ (무주택기간) 「지방세 세목별 과세증명서」(해지 시 제출, 전국단위 주민센터 
발급본, 우대금리 적용기간 증빙용)
  * 비과세 위한 연소득 및 무주택세대주 요건 증빙은 조세특례제한법을 따름 
납입방식
‣ 원금 1,500만원까지 자유롭게 납입 후, 월2만〜100만원 납입 가능
  * 청년도약계좌, 청년희망적금의 만기수령금을 일시납하는 경우 5,000만원까지 
일시납 가능
가입기간
‣ 2024.2.21. ~ 2025.12.31. (일몰제 적용)


## PDF 문서 로더 텍스트 분류

- PDF 문서에 Q 와 A 가 각각 이미지로 구성되어있고, FAQ 형식이 고정된 텍스트가 아니어서 비정형 문서의 정보를 추출하기 힘든 상황
- LLM 으로 문서 청킹 및 분류 요청
- 분류된 문서 벡터 저장소 저장

In [11]:
import os
from typing import List, Optional
from pydantic import BaseModel, Field

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# ==========================================
# 1. 추출할 데이터 구조 정의 (Pydantic)
# ==========================================
class CleanQAPair(BaseModel):
    """정제된 단일 Q&A 데이터"""
    question_id: Optional[int] = Field(default=None, description="질문 번호 (텍스트에서 유추 가능할 경우 숫자만 추출, 없으면 null)")
    chapter: str = Field(description="이 질문이 속한 대주제 (예: 청약통장, 특별공급, 무주택자 기준 등). 텍스트 문맥을 보고 직접 분류하세요.")
    question: str = Field(description="사용자가 묻는 형태의 완전한 질문 문장")
    answer: str = Field(description="질문에 대한 명확하고 완전한 답변 문장")

class DocumentExtraction(BaseModel):
    """페이지나 청크에서 추출된 전체 Q&A 목록"""
    qa_list: List[CleanQAPair] = Field(description="추출된 Q&A 쌍의 목록. 노이즈만 있는 페이지라면 빈 배열을 반환하세요.")
    ignored_noise_summary: str = Field(description="이 텍스트에서 무시한 데이터(목차, 법령 참고, 페이지 번호 등)가 무엇인지 1문장으로 요약")

# ==========================================
# 2. LLM 추출 파이프라인 클래스
# ==========================================

class LLMDataRefiner:
    def __init__(self):
        # 전처리용이므로 가성비가 좋은 모델 사용 (temperature는 0으로 고정하여 환각 방지)
        self.llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
        
        # Pydantic 모델을 LLM에 바인딩하여 무조건 JSON 구조로만 응답하게 강제
        self.extractor = self.llm.with_structured_output(DocumentExtraction)
        
        # 정제 지침 프롬프트
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", """당신은 주택청약 문서 데이터 정제 전문가입니다.
            제공된 [Raw Text]에서 유효한 '질문(Q)'과 '답변(A)' 쌍을 찾아내어 구조화된 JSON으로 추출하세요.
            
            [엄격한 지침]
            1. '목차', '가. 주요내용', '관련 법령', '페이지 번호', '단순 공지사항' 등은 철저히 무시하세요.
            2. Q나 A 마커가 깨져있거나 없더라도 문맥상 질문과 답변이라면 추출하세요.
            3. 답변이 끊겨있다면 문맥을 부드럽게 이어 완전한 문장으로 만드세요.
            4. 해당 텍스트 블록에 유효한 Q&A가 전혀 없다면 qa_list를 빈 배열([])로 반환하세요.
            5. "1. 청약자격" , "2. 청약통장" ~~ "5. 주택공급절차" 과 같이 대단원별로 QA 질의응답을 추출하세요.
            """),
            ("human", "<raw_text>\n{text}</<raw_text>")
        ])
        
        self.chain = self.prompt | self.extractor

    def refine_text_chunk(self, raw_text: str) -> DocumentExtraction:
        """단일 텍스트 덩어리를 정제합니다."""
        print("🤖 LLM이 텍스트에서 데이터를 추출 중입니다...")
        return self.chain.invoke({"text": raw_text})


In [12]:
body_text = "\n" + "\n".join([doc.page_content for doc in raw_documents[start_page_idx:]])


# 파이프라인 초기화 및 실행
refiner = LLMDataRefiner()
result = refiner.refine_text_chunk(body_text)

print("\n" + "="*50)
print("✅ [추출 완료] LLM이 파악한 노이즈 요약:")
print(f"-> {result.ignored_noise_summary}")
print("="*50)

# 추출된 JSON 데이터를 LangChain Document로 변환
final_docs = []
for qa in result.qa_list:
    content = f"[주제: {qa.chapter}]\n질문: {qa.question}\n\n답변: {qa.answer}"
    
    doc = Document(
        page_content=content,
        metadata={
            "question_id": qa.question_id if qa.question_id else 0,
            "chapter": qa.chapter
        }
    )
    final_docs.append(doc)

# 결과 확인
for idx, doc in enumerate(final_docs, 1):
    print(f"\n[정제된 Document {idx}]")
    print(doc.page_content)
    print(f"메타데이터: {doc.metadata}")

🤖 LLM이 텍스트에서 데이터를 추출 중입니다...

✅ [추출 완료] LLM이 파악한 노이즈 요약:
-> 문서에 포함된 목차, 법령, 페이지 번호, 단순 공지사항 등은 무시하였습니다.

[정제된 Document 1]
[주제: 1. 청약자격]
질문: 경기도 과천시에서 공급되는 주택의 해당 주택건설지역의 범위는?

답변: 해당 주택건설지역이란 특별시ㆍ광역시ㆍ특별자치시ㆍ특별자치도(관할 구역 안에 지방자치단체인 시ㆍ군이 없는 특별자치도를 말한다) 또는 시ㆍ군의 행정구역을 말합니다. 따라서, 경기도 과천시에서 공급하는 주택의 경우 과천시가 해당 주택건설지역에 해당됩니다. 참고로, 서울특별시에서 공급되는 주택의 경우 서울특별시 전역, 인천광역시의 경우 인천광역시 전역이 해당 주택건설지역에 해당됩니다.
메타데이터: {'question_id': 1, 'chapter': '1. 청약자격'}

[정제된 Document 2]
[주제: 1. 청약자격]
질문: 해당 주택건설지역에 거주하고 있지 않다면 청약신청이 불가능한지?

답변: 해당 주택건설지역에 거주하고 있지 않더라도 청약가능지역에서 공급되는 주택에 청약 신청이 가능하나, 같은 순위에서는 해당 주택건설지역의 거주자가 우선하여 주택을 공급받게 됩니다. 다만, 수도권 대규모 택지개발지구 등에서 주택이 공급되는 경우 일정 비율의 주택에 대해서는 해당 주택건설지역 거주자와 동등한 자격으로 주택을 공급받을 기회를 가지게 됩니다.
메타데이터: {'question_id': 2, 'chapter': '1. 청약자격'}

[정제된 Document 3]
[주제: 1. 청약자격]
질문: 해당 지역에 거주하고 있으나, 우선공급을 위한 거주기간을 충족하지 못하는 경우 청약신청 지역은?

답변: 입주자모집공고일 현재 해당지역에 거주하고 있으나 우선공급을 위한 거주기간을 충족하지 못하는 경우에는 기타지역으로 청약을 신청하여야 하며, 거주기간을 충족하지 못함에도 해당지역 우선공급으로 청약을 신청하여 당첨된 경우에는 당첨이 취소되고 부적격 처리되어 일정기

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import time


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000,
    chunk_overlap=300,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_text(body_text)
print(f"📦 전체 텍스트가 {len(chunks)}개의 청크(Chunk)로 분할되었습니다.\n")

# 파이프라인 초기화
refiner = LLMDataRefiner()
final_docs = []

# 중복 방지용 Set (오버랩 구간에서 동일한 질문이 두 번 추출되는 것 방지)
seen_questions = set() 

# 3. 분할된 청크를 순회하며 부분부분 추출
for i, chunk in enumerate(chunks):
    print(f"⏳ [{i+1}/{len(chunks)}] 번째 청크 처리 중... (길이: {len(chunk)}자)")
    
    try:
        # LLM 정제 파이프라인 통과
        result = refiner.refine_text_chunk(chunk)
        
        print(f"  -> 🟢 추출된 Q&A: {len(result.qa_list)}개")
        print(f"  -> 🗑️ 무시된 노이즈: {result.ignored_noise_summary}")
        
        # 추출된 데이터를 Document로 변환
        for qa in result.qa_list:
            # 질문 내용이 이미 추출된 적이 없는 경우에만 추가
            if qa.question not in seen_questions:
                seen_questions.add(qa.question)
                
                content = f"[주제: {qa.chapter}]\n질문: {qa.question}\n\n답변: {qa.answer}"
                
                doc = Document(
                    page_content=content,
                    metadata={
                        "question_id": qa.question_id if qa.question_id else 0,
                        "chapter": qa.chapter,
                        "source_chunk_idx": i+1 # 디버깅용 출처 청크 번호
                    }
                )
                final_docs.append(doc)
                
    except Exception as e:
        print(f"  [!] {i+1}번째 청크 처리 중 오류 발생: {e}")
        
    # API Rate Limit (호출 빈도 제한) 방지를 위해 1초 대기 (주피터 환경에서 안정적)
    time.sleep(1)

# 4. 최종 결과 확인
print("\n" + "="*50)
print(f"✅ [최종 완료] 총 {len(final_docs)}개의 고품질 Q&A Document가 성공적으로 추출되었습니다!")
print("="*50)

# 추출된 데이터 샘플 3개만 출력 확인
for idx, doc in enumerate(final_docs[:3], 1):
    print(f"\n[정제된 Document {idx}]")
    print(doc.page_content)
    print(f"메타데이터: {doc.metadata}")

📦 전체 텍스트가 79개의 청크(Chunk)로 분할되었습니다.

⏳ [1/79] 번째 청크 처리 중... (길이: 2976자)
🤖 LLM이 텍스트에서 데이터를 추출 중입니다...
  -> 🟢 추출된 Q&A: 7개
  -> 🗑️ 무시된 노이즈: 목차, 법령 조문 전문, 페이지 번호, 단순 공지사항 등은 무시하고 청약자격 관련 주요 내용과 질의응답만 추출하였습니다.
⏳ [2/79] 번째 청크 처리 중... (길이: 2958자)
🤖 LLM이 텍스트에서 데이터를 추출 중입니다...
  -> 🟢 추출된 Q&A: 4개
  -> 🗑️ 무시된 노이즈: 제4조, 제34조 등 법령 조항과 대규모 택지개발지구 우선공급 관련 내용, 거주기간 산정 기준, 국외 체류 예외사항 등은 단순 법령 설명 및 공지사항으로 판단되어 무시하고, 청약자격 중 청약신청지역 및 우선공급 관련 실질적 질문과 답변만 추출함.
⏳ [3/79] 번째 청크 처리 중... (길이: 2989자)
🤖 LLM이 텍스트에서 데이터를 추출 중입니다...
  -> 🟢 추출된 Q&A: 5개
  -> 🗑️ 무시된 노이즈: 문서에 목차, 관련 법령 조문, 페이지 번호, 단순 공지사항 등 무의미한 내용이 포함되어 있으나 1문장으로 요약함.
⏳ [4/79] 번째 청크 처리 중... (길이: 2989자)
🤖 LLM이 텍스트에서 데이터를 추출 중입니다...
  -> 🟢 추출된 Q&A: 11개
  -> 🗑️ 무시된 노이즈: 문서에 포함된 목차, 관련 법령, 페이지 번호, 단순 공지사항 등은 무시하고 해외 체류기간과 청약자격 관련 Q&A만 추출하였습니다.
⏳ [5/79] 번째 청크 처리 중... (길이: 2959자)
🤖 LLM이 텍스트에서 데이터를 추출 중입니다...
  -> 🟢 추출된 Q&A: 3개
  -> 🗑️ 무시된 노이즈: 입주자모집공고일 기준 단신부임 여부 판단, 청약통장 종류 및 가입 대상, 저축 방식, 저축 금액, 주택청약종합저축 및 청년주택드림청약통장 상품 개요, 관련 법령 및 페이지 번호 등은 무시함.
⏳ [6/79] 번째

In [14]:
#문서 저장
output_file = "./data/housing_faq_data_2024.json"

with open(output_file, 'w', encoding='utf-8-sig') as f:
    json.dump([doc.model_dump() for doc in final_docs], f, indent=2, ensure_ascii=False)  # 한글이 유니코드로 변환되지 않도록 설정
    
print(f"포맷팅된 문서를 {output_file}에 저장했습니다.")

포맷팅된 문서를 ./data/housing_faq_data_2024.json에 저장했습니다.


In [21]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 문서 벡터 저장
vector_store = Chroma.from_documents(  
    documents=final_docs,
    embedding=embeddings,
    collection_name="housing_faq_db",
    persist_directory="./chroma_db",
)

In [22]:
vector_store._collection.count()

509

In [23]:
from langchain_core.vectorstores import VectorStoreRetriever

mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"fetch_k": 10, "k": 3, "lambda_mult": 0.5}
)

In [24]:
result = mmr_retriever.invoke("청약통장을 종합저축으로 전환가능한지?")

print(result)

[Document(id='c2d2df78-fac7-41ae-b1d1-0cdfb1816c60', metadata={'chapter': '1. 청약자격', 'source_chunk_idx': 6, 'question_id': 1}, page_content='[주제: 1. 청약자격]\n질문: 기존 청약통장을 주택청약종합저축으로 전환할 수 있나요?\n\n답변: 기존 청약통장(청약저축, 청약부금, 청약예금)에서 주택청약종합저축으로의 전환은 불가능하며, 기존 청약통장을 해지한 후 신규 가입으로만 가능합니다.'), Document(id='53b03256-49f8-48ee-85ac-048ae3a64f0f', metadata={'question_id': 174, 'chapter': 'Ⅱ. 일반공급', 'source_chunk_idx': 28}, page_content='[주제: Ⅱ. 일반공급]\n질문: 통장기간 합산 시 해당 통장을 사용한 것으로 간주하는지?\n\n답변: 배우자의 통장기간을 합산하더라도 청약상의 다른 부분에는 영향이 없으므로 통장을 사용한 것으로 간주하지 않습니다.'), Document(id='153d766a-40aa-41ba-aa59-7a8e26348614', metadata={'chapter': '1. 청약자격', 'source_chunk_idx': 6, 'question_id': 3}, page_content='[주제: 1. 청약자격]\n질문: 이미 납입한 회차의 예치금을 추후에 수정할 수 있나요?\n\n답변: 한번 입금된 금액과 회차는 정정이 불가합니다. 예를 들어 한 회차에 2만원 납입 후, 추후에 추가 납입하여 해당 회차 납입금을 10만원으로 정정할 수는 없습니다.')]


In [25]:
from pydantic import BaseModel, Field
from typing import Optional, Literal

class MetadataFilter(BaseModel):
    """Chroma DB 메타데이터 필터 조건 (2024년 데이터베이스 규격 맞춤)"""
    # 💡 [개선]: keyword 대신 2024 데이터의 'chapter' 메타데이터 매핑
    chapter: Optional[str] = Field(
        default=None, 
        description="검색 대상 주제 영역 (예: '1. 청약자격', 'Ⅰ. 청약자격(공통)', 'Ⅰ. 청약자격(공통) - 나. 청약신청지역 및 우선공급')"
    )
    chapter_operator: Optional[Literal["$eq", "$ne"]] = Field(
        default=None, description="주제 비교 연산자 ($eq: 일치, $ne: 불일치)"
    )
    # 질문 ID 범위 (하한)
    question_id_min: Optional[int] = Field(default=None, description="질문 ID 최소값")
    question_id_min_operator: Optional[Literal["$gt", "$gte"]] = Field(
        default=None, description="최소값 연산자 ($gt: 초과, $gte: 이상)"
    )
    # 질문 ID 범위 (상한)
    question_id_max: Optional[int] = Field(default=None, description="질문 ID 최대값")
    question_id_max_operator: Optional[Literal["$lt", "$lte"]] = Field(
        default=None, description="최대값 연산자 ($lt: 미만, $lte: 이하)"
    )
    # 논리 연산자
    logical_operator: Optional[Literal["$and", "$or"]] = Field(
        default="$and", description="복합 조건 결합 연산자"
    )

In [27]:
from dataclasses import dataclass

@dataclass
class SearchResult:
    context: str
    source_documents: Optional[List[Document]]

class AnswerEvaluation(BaseModel):
    """답변 품질 평가 결과 모델"""
    is_valid: bool = Field(description="답변이 문맥(Context)에 기반하여 질문에 충분하고 정확하게 대답하고 있는지 여부")
    reason: str = Field(description="평가 결과에 대한 구체적인 이유 (예: '문맥에 관련 정보가 없음', '정확히 답변함' 등)")

In [ ]:
import sqlite3
import json
from typing import List
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage

class SQLiteTrimmedAndSummarizedHistory(BaseChatMessageHistory):
    """
    SQLite를 사용하여 메시지 트리밍과 대화 요약을 영구 저장하는 대화 히스토리 클래스
    """
    def __init__(self, session_id: str, db_path: str = "chat_history_faq.db", max_messages: int = 4, llm=None):
        self.session_id = session_id
        self.db_path = db_path
        self.max_messages = max_messages
        self.llm = llm
        self._create_tables()
    def _create_tables(self):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS messages (
                id INTEGER PRIMARY KEY AUTOINCREMENT, session_id TEXT, message_type TEXT, content TEXT, metadata TEXT, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        """)
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS summaries (
                session_id TEXT PRIMARY KEY, summary TEXT, updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        """)
        conn.commit()
        conn.close()
    def get_summary(self) -> str:
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT summary FROM summaries WHERE session_id = ?", (self.session_id,))
        row = cursor.fetchone()
        conn.close()
        return row[0] if row else ""
    def save_summary(self, summary: str):
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute("""
            INSERT INTO summaries (session_id, summary, updated_at) VALUES (?, ?, CURRENT_TIMESTAMP)
            ON CONFLICT(session_id) DO UPDATE SET summary = excluded.summary, updated_at = CURRENT_TIMESTAMP
        """, (self.session_id, summary))
        conn.commit()
        conn.close()
    @property
    def messages(self) -> List[BaseMessage]:
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute("SELECT message_type, content FROM messages WHERE session_id = ? ORDER BY id ASC", (self.session_id,))
        loaded_messages = []
        for msg_type, content in cursor.fetchall():
            if msg_type == "HumanMessage":
                loaded_messages.append(HumanMessage(content=content))
            elif msg_type == "AIMessage":
                loaded_messages.append(AIMessage(content=content))
        conn.close()
        return loaded_messages
    def add_messages(self, messages: List[BaseMessage]) -> None:
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        for msg in messages:
            cursor.execute("INSERT INTO messages (session_id, message_type, content) VALUES (?, ?, ?)", 
                           (self.session_id, msg.__class__.__name__, msg.content))
        conn.commit()
        
        # 트리밍 및 요약 로직
        cursor.execute("SELECT id, message_type, content FROM messages WHERE session_id = ? ORDER BY id ASC", (self.session_id,))
        all_rows = cursor.fetchall()
        conn.close()
        
        if len(all_rows) > self.max_messages:
            num_to_delete = len(all_rows) - self.max_messages
            to_delete_rows = all_rows[:num_to_delete]
            
            # 요약할 텍스트 추출
            delete_chats = [f"{'User' if r[1]=='HumanMessage' else 'AI'}: {r[2]}" for r in to_delete_rows]
            new_chats_to_summarize = "\n".join(delete_chats)
            
            existing_summary = self.get_summary()
            summary_prompt = (
                f"기존 요약본:\n{existing_summary or '없음'}\n\n"
                f"새로 추가된 대화:\n{new_chats_to_summarize}\n\n"
                "위 내용을 바탕으로 전체 맥락이 이어지도록 간결한 통합 요약(한글)을 작성해 주세요."
            )
            
            summary_msg = self.llm.invoke([
                SystemMessage(content="당신은 대화의 맥락을 요약하여 기록하는 비서입니다."),
                HumanMessage(content=summary_prompt)
            ])
            self.save_summary(summary_msg.content)
            
            # 데이터 삭제
            delete_ids = [row[0] for row in to_delete_rows]
            conn = sqlite3.connect(self.db_path)
            cursor = conn.cursor()
            cursor.execute(f"DELETE FROM messages WHERE id IN ({','.join(['?'] * len(delete_ids))})", delete_ids)
            conn.commit()
            conn.close()
    def clear(self) -> None:
        """현재 세션의 대화 상세 이력 및 요약본을 데이터베이스에서 전부 비웁니다."""
        conn = sqlite3.connect(self.db_path)
        cursor = conn.cursor()
        cursor.execute("DELETE FROM messages WHERE session_id = ?", (self.session_id,))
        cursor.execute("DELETE FROM summaries WHERE session_id = ?", (self.session_id,))
        conn.commit()
        conn.close()
        print(f"🧹 [세션 {self.session_id}] 대화 이력이 비워졌습니다.")

In [32]:
from langchain_core.language_models import BaseChatModel
from langchain_core.vectorstores import VectorStoreRetriever
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from typing import List, Optional, Generator, Dict

# ==========================================
# RAG 시스템 클래스 개선 구현 (2024 데이터셋 필터 지원)
# ==========================================
class HousingRAGSystem:
    def __init__(
            self, 
            llm: BaseChatModel, 
            eval_llm: BaseChatModel,
            retriever: VectorStoreRetriever
        ):
        self.llm = llm or ChatOpenAI(model="gpt-4.1-mini", temperature=0)
        self.eval_llm = eval_llm or ChatOpenAI(model="gpt-4.1", temperature=0)
        self.retriever = retriever
        self.vectorstore = retriever.vectorstore
        
        # 💡 [개선]: 추출 규칙에 chapter(주제)를 타겟하도록 시스템 프롬프트 업데이트
        self.filter_extractor = self.eval_llm.with_structured_output(MetadataFilter)
        self.filter_prompt = ChatPromptTemplate.from_messages([
            ("system", """사용자 쿼리에서 Chroma DB 검색 필터 조건을 추출합니다.
## 추출 규칙
### 주제 영역 (chapter)
- 질문이 속한 대단원이나 주제 영역을 분류하여 추출합니다.
- 예시 카테고리:
  - "1. 청약자격"
  - "Ⅰ. 청약자격(공통) - 나. 청약신청지역 및 우선공급"
  - "Ⅰ. 청약자격(공통)"
- chapter_operator: 일반적으로 "$eq" 사용
### 질문 ID 범위 (question_id)
- "N번 이상": question_id_min=N, question_id_min_operator="$gte"
- "N번 이하": question_id_max=N, question_id_max_operator="$lte"
- "N~M번 사이": 최소값과 최대값 모두 설정
해당 조건이 명시적으로 언급되지 않았거나 불확실하면 필드를 비워(null) 두세요.
"""),
            ("human", "{question}")
        ])
    # 💡 [개선]: build_chroma_filter에서 keyword 대신 chapter 조건 결합
    def _build_chroma_filter(self, filter_model: MetadataFilter) -> Optional[Dict]:
        """추출된 조건을 Chroma DB 규격의 필터로 변환"""
        conditions = []
        if filter_model.chapter:
            conditions.append({"chapter": {filter_model.chapter_operator: filter_model.chapter}})
        if filter_model.question_id_min is not None:
            conditions.append({"question_id": {filter_model.question_id_min_operator: filter_model.question_id_min}})
        if filter_model.question_id_max is not None:
            conditions.append({"question_id": {filter_model.question_id_max_operator: filter_model.question_id_max}})
            
        if not conditions: return None
        return conditions[0] if len(conditions) == 1 else {filter_model.logical_operator: conditions}
    def _format_docs(self, docs: List[Document]) -> str:
        return "\n\n".join(doc.page_content for doc in docs)
    
    def _format_source_documents(self, docs: Optional[List[Document]]) -> str:
        if not docs:
            return "\n\nℹ️ 관련 문서를 찾을 수 없습니다."
        
        formatted_docs = []
        for i, doc in enumerate(docs, 1):
            metadata = doc.metadata if hasattr(doc, 'metadata') else {}
            source_info = []
            
            if 'question_id' in metadata:
                source_info.append(f"ID: {metadata['question_id']}")
            if 'chapter' in metadata:
                source_info.append(f"주제: {metadata['chapter']}")
            if 'source_chunk_idx' in metadata:
                source_info.append(f"청크 인덱스: {metadata['source_chunk_idx']}")
                
            formatted_docs.append(
                f"**[참조 문서 {i}]** {' | '.join(source_info) if source_info else '출처 정보 없음'}\n"
                f"> {doc.page_content.replace('\n', ' ')}"
            )
        return "\n\n**[근거 문서]**\n" + "\n\n".join(formatted_docs)
    def _check_relevance(self, docs: List[Document], question: str) -> List[Document]:
        relevant_docs = []
        if not docs: return relevant_docs
            
        prompt = ChatPromptTemplate.from_messages([
            ("system", "주어진 컨텍스트가 질문에 답변하는데 필요한 정보를 포함하고 있는지 평가하세요.\n\n"
                       "직접적으로 포함하거나 논리적으로 추론 가능하다면 'Yes', 아니면 'No'로만 답변하세요."),
            ("human", "<context>\n{context}</context>\n\n<question>\n{question}</question>")
        ])
        chain = prompt | self.eval_llm | StrOutputParser()
        print("\n🔍 [관련성 평가 시작]")
        for doc in docs:
            result = chain.invoke({"context": doc.page_content, "question": question}).lower()
            if "yes" in result:
                relevant_docs.append(doc)
            else:
                print(f"  - 문서 배제됨: {doc.page_content[:30]}...")
                
        return relevant_docs
    def search_documents(self, question: str) -> SearchResult:
        try:
            # 1. 자연어 질문에서 동적 필터 조건 추출
            extracted_filter = self.filter_extractor.invoke(self.filter_prompt.format(question=question))
            chroma_filter = self._build_chroma_filter(extracted_filter)
            print(f"🎯 [적용된 필터]: {chroma_filter}")
            
            # 2. 필터 적용하여 문서 검색
            search_kwargs = {"k": 4}
            if chroma_filter: search_kwargs["filter"] = chroma_filter
            retriever = self.vectorstore.as_retriever(search_kwargs=search_kwargs)
            
            raw_docs = retriever.invoke(question)
            print(f"📄 [1차 검색 문서 개수]: {len(raw_docs)}")
            
            # 3. LLM 기반 관련성 검증
            relevant_docs = self._check_relevance(raw_docs, question) 
            print(f"✅ [최종 유효 문서 개수]: {len(relevant_docs)}")
            
            return SearchResult(
                context=self._format_docs(relevant_docs) if relevant_docs else "관련 문서를 찾을 수 없습니다.",
                source_documents=relevant_docs,
            )
        except Exception as e:
            print(f"문서 검색 중 오류 발생: {e}")
            return SearchResult(context="문서 검색 중 오류가 발생했습니다.", source_documents=None)
            
    def generate_answer(self, message: str, history: List) -> Generator[str, None, None]:
        # 1. 문서 검색 진행
        search_result = self.search_documents(message)
        
        if not search_result.source_documents:
            yield "⚠️ 죄송합니다. 제공된 FAQ 문서 내에서 질문에 대한 명확한 답변이나 근거를 찾을 수 없습니다."
            return
        
        # 2. SQLite 기반의 Trimmed & Summarized History 객체 연결
        # 데모 구동용 고정 session_id를 할당합니다.
        session_id = "faq_chatbot_session"
        db_history = SQLiteTrimmedAndSummarizedHistory(
            session_id=session_id,
            max_messages=4,  # 원본 보존할 최근 메시지 턴 수 (질문2개+답변2개)
            llm=self.llm
        )
        
        # SQLite에서 대화 이력 및 요약본 가져오기
        existing_summary = db_history.get_summary()
        
        # 3. LLM에 입력할 전체 메시지 목록(Context 포함) 빌드
        messages = []
        
        # 시스템 프롬프트에 '이전 대화 요약' 정보 삽입
        system_content = """당신은 주택청약 전문 상담가입니다. 다음 지침을 엄격히 따르세요:
1. 제공된 [문서들]의 내용만을 기반으로 답변하세요.
2. 문서에 명확한 근거가 없는 내용은 "근거 없음"이라고 답변하세요.
3. 추측이나 일반적인 사전 지식을 섞어서 사용하지 마세요."""
        if existing_summary:
            system_content += f"\n\n[이전 대화 요약]\n{existing_summary}"
            
        messages.append(SystemMessage(content=system_content))
        
        # 최근 상세 대화 4개 추가
        messages.extend(db_history.messages)
        
        # 현재 컨텍스트와 질문 추가
        current_human_content = f"<context>\n{search_result.context}\n</context>\n\n<question>{message}</question>"
        messages.append(HumanMessage(content=current_human_content))
        
        full_answer = ""
        try:
            # 4. 스트리밍 실행 (chain 대신 llm을 직접 활용하여 메시지 리스트 전달)
            for chunk in self.llm.stream(messages):
                # ChatOpenAI stream은 AIMessageChunk 객체를 반환하므로 .content로 추출합니다.
                content_chunk = chunk.content if hasattr(chunk, 'content') else chunk
                full_answer += content_chunk
                yield full_answer
            
            # 출처 추가
            sources = self._format_source_documents(search_result.source_documents)
            final_response = f"{full_answer}\n\n---\n{sources}"
            yield final_response
            
            # 5. [핵심] 성공적으로 스트리밍이 완료되면 대화 히스토리를 DB에 저장
            db_history.add_messages([
                HumanMessage(content=message),
                AIMessage(content=full_answer)
            ])
            
        except Exception as e:
            yield f"답변 생성 중 오류가 발생했습니다: {str(e)}"

In [38]:
# 1. RAG 시스템 인스턴스 초기화
rag_system = HousingRAGSystem(
    llm=ChatOpenAI(model="gpt-4.1-mini", temperature=0),  
    eval_llm=ChatOpenAI(model="gpt-4.1-mini", temperature=0),
    retriever=mmr_retriever,
)
print("\n--- 테스트 1: 정상적인 질문 (DB에 정보가 있는 경우) ---")

generator1 = rag_system.generate_answer(
    message="무주택세대구성원 이란?", 
    history=[]
)

yielded_outputs1 = list(generator1)
final_response1 = yielded_outputs1[-1] if yielded_outputs1 else "답변을 생성하지 못했습니다."
print(f"\ 테스트1 답변:\n{final_response1}\n")


print("\n--- 테스트 2: 비정상적 질문 (DB에 정보가 없는 경우) ---")
generator2 = rag_system.generate_answer(
    message="한국 1인당 GDP에 대해서 알려주세요", 
    history=[]
)
yielded_outputs2 = list(generator2)
final_response2 = yielded_outputs2[-1] if yielded_outputs2 else "답변을 생성하지 못했습니다."

print(f"\n 테스트2 답변:\n{final_response2}\n")

<>:16: SyntaxWarning: invalid escape sequence '\ '
<>:16: SyntaxWarning: invalid escape sequence '\ '
C:\Users\JSPark\AppData\Local\Temp\ipykernel_36260\237980511.py:16: SyntaxWarning: invalid escape sequence '\ '
  print(f"\ 테스트1 답변:\n{final_response1}\n")



--- 테스트 1: 정상적인 질문 (DB에 정보가 있는 경우) ---
🎯 [적용된 필터]: {'chapter': {'$eq': '1. 청약자격'}}
📄 [1차 검색 문서 개수]: 4

🔍 [관련성 평가 시작]
  - 문서 배제됨: [주제: 1. 청약자격]
질문: 무주택자인 아내가 유주...
  - 문서 배제됨: [주제: 1. 청약자격]
질문: 부부가 모두 무주택자이...
  - 문서 배제됨: [주제: 1. 청약자격]
질문: 주택을 소유하고 있는 ...
✅ [최종 유효 문서 개수]: 1
\ 테스트1 답변:
무주택세대구성원이란 청약신청자 및 세대원 전원이 주택을 소유하고 있지 않은 세대의 구성원(세대주 포함)을 말합니다.

---


**[근거 문서]**
**[참조 문서 1]** ID: 47 | 주제: 1. 청약자격 | 청크 인덱스: 9
> [주제: 1. 청약자격] 질문: 무주택세대구성원이란 무엇인가요?  답변: 무주택세대구성원이란 청약신청자 및 세대원 전원이 주택을 소유하고 있지 않은 세대의 구성원(세대주 포함)을 말합니다.


--- 테스트 2: 비정상적 질문 (DB에 정보가 없는 경우) ---
🎯 [적용된 필터]: None
📄 [1차 검색 문서 개수]: 4

🔍 [관련성 평가 시작]
  - 문서 배제됨: [주제: Ⅳ. 소득산정]
질문: 국민연금의 기준소득월액...
  - 문서 배제됨: [주제: Ⅳ. 소득산정]
질문: 사업자인데 전년도 소득...
  - 문서 배제됨: [주제: 특별공급 및 우선공급]
질문: 외국인 배우자가...
  - 문서 배제됨: [주제: Ⅳ. 소득산정]
질문: 상시근로자의 소득산정 ...
✅ [최종 유효 문서 개수]: 0

 테스트2 답변:
⚠️ 죄송합니다. 제공된 FAQ 문서 내에서 질문에 대한 명확한 답변이나 근거를 찾을 수 없습니다.



In [39]:
import gradio as gr


rag_system = HousingRAGSystem(
    llm=ChatOpenAI(model="gpt-4.1-mini", temperature=0),  
    eval_llm=ChatOpenAI(model="gpt-4.1-mini", temperature=0),
    retriever=mmr_retriever,
)

# Gradio ChatInterface 구성
demo = gr.ChatInterface(
    fn=rag_system.generate_answer,
    title="🏢 스마트 주택청약 챗봇",
    description="최신 주택청약 FAQ를 기반으로 답변해 드립니다. 질문을 입력해 보세요!",
    examples=[
        "청약신청 지역을 판단하는 기준은 무엇인가요?",
        "주택 공급대상은 어떻게 정해지나요?",
        "수도권 투기과열지구에서 주택 우선공급을 위한 거주기간 요건은 어떻게 되나요?" 
    ],
    fill_height=True
)

demo.launch() 


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


🎯 [적용된 필터]: {'chapter': {'$eq': 'Ⅰ. 청약자격(공통) - 나. 청약신청지역 및 우선공급'}}
📄 [1차 검색 문서 개수]: 4

🔍 [관련성 평가 시작]
  - 문서 배제됨: [주제: Ⅰ. 청약자격(공통) - 나. 청약신청지역 및...
  - 문서 배제됨: [주제: Ⅰ. 청약자격(공통) - 나. 청약신청지역 및...
  - 문서 배제됨: [주제: Ⅰ. 청약자격(공통) - 나. 청약신청지역 및...
  - 문서 배제됨: [주제: Ⅰ. 청약자격(공통) - 나. 청약신청지역 및...
✅ [최종 유효 문서 개수]: 0
🎯 [적용된 필터]: None
📄 [1차 검색 문서 개수]: 4

🔍 [관련성 평가 시작]
  - 문서 배제됨: [주제: 1. 청약자격]
질문: 같은 순위에서 주택 공...
  - 문서 배제됨: [주제: Ⅴ. 주택공급절차]
질문: 무순위 청약으로 주...
✅ [최종 유효 문서 개수]: 2
🎯 [적용된 필터]: {'chapter': {'$eq': '1. 청약자격'}}
📄 [1차 검색 문서 개수]: 4

🔍 [관련성 평가 시작]
  - 문서 배제됨: [주제: 1. 청약자격]
질문: 주택공급에 관한 규칙 ...
  - 문서 배제됨: [주제: 1. 청약자격]
질문: 청약신청 지역을 판단하...
  - 문서 배제됨: [주제: 1. 청약자격]
질문: 주택이 아닌 다른 용도...
  - 문서 배제됨: [주제: 1. 청약자격]
질문: 재당첨 대상 주택 및 ...
✅ [최종 유효 문서 개수]: 0
🎯 [적용된 필터]: None
📄 [1차 검색 문서 개수]: 4

🔍 [관련성 평가 시작]
  - 문서 배제됨: [주제: Ⅳ. 소득산정]
질문: 입주자모집공고일 이후 ...
  - 문서 배제됨: [주제: Ⅳ. 소득산정]
질문: 농업 종사자이나 현재 ...
  - 문서 배제됨: [주제: Ⅳ. 소득산정]
질문: 상시근로자의 소득산정 ...
✅ [최종 유효 문서 개수]: 1


In [132]:
# Gradio 인터페이스 종료
demo.close()

Closing server running on port: 7863
